# 02 — Fine-tune SegFormer on FoodSeg103 (GPU job 1)

Trains the on-device food segmenter. Every public FoodSeg103 checkpoint measures at mIoU ≤ 0.05 (docs/MODELS.md), so we train our own.

- **Private repo:** the clone cell needs a GitHub token — add it once as a Colab secret named `GH_TOKEN` (🔑 in the left sidebar, "Add new secret", paste a PAT with `repo` scope, toggle notebook access on). If the repo is public, no token is needed.
- **Runtime:** pick an A100/H100 runtime. The batch auto-shrinks on a small GPU (T4 → batch 8) so it won't OOM, but a T4 will be slow — H100 is B0 ~2–3 h, T4 many hours.
- **Targets:** mIoU ≥ 0.25 (mit-b0) / ≥ 0.32 (mit-b1)
- **Persistence:** trained checkpoints land in `OUT` on Drive; the dataset and backbone caches are local (they re-download in seconds). Notebook 01 is NOT required for this one.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, torch
DRIVE_ROOT = '/content/drive/MyDrive/physics-powered-portion-estimation'

# HF caches stay on LOCAL disk. The datasets library memory-maps its Arrow
# files, and mmap over Google Drive's FUSE mount fails — so the dataset and
# the (tiny) pretrained backbone live locally and re-download in seconds on a
# fresh VM. Only the trained checkpoints (OUT, below) persist to Drive.
os.environ['HF_HOME'] = '/content/hf-cache'
os.environ['HF_DATASETS_CACHE'] = '/content/hf-datasets'

MODEL = 'nvidia/mit-b0'   # or 'nvidia/mit-b1'
EPOCHS = 60
# Auto-size the batch to the GPU so a small card doesn't OOM:
# 32 on A100/H100 (>24 GB VRAM), 8 on a T4 (16 GB).
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
BATCH = 32 if vram_gb > 24 else 8
OUT = f"{DRIVE_ROOT}/checkpoints/segformer-{MODEL.split('/')[-1]}-food"
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f'GPU: {gpu} · {vram_gb:.0f} GB · batch {BATCH}')
if vram_gb and vram_gb < 24:
    print('WARNING: small GPU — training will be slow. Prefer an A100/H100 runtime.')

!nvidia-smi | head -12

In [ ]:
# Clone the repo. Private repo → add your GitHub token as a Colab secret named
# GH_TOKEN (🔑 left sidebar). Public repo → no token needed. Fails LOUD if the
# clone didn't produce the code (no more silent 2>/dev/null).
import os, subprocess
token = ''
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN') or ''
except Exception:
    pass
repo = 'github.com/Hakeem-Hannoon/physics-powered-portion-estimation.git'
url = f'https://x-access-token:{token}@{repo}' if token else f'https://{repo}'
subprocess.run(['rm', '-rf', '/content/ppe'])
r = subprocess.run(['git', 'clone', '--depth', '1', url, '/content/ppe'],
                   capture_output=True, text=True)
assert os.path.isdir('/content/ppe/model'), (
    (r.stderr.replace(token, '***') if token else r.stderr) +
    '\nClone failed — add a GH_TOKEN Colab secret (🔑) or make the repo public.')
print('repo ready')

%pip -q install transformers datasets evaluate accelerate timm

In [ ]:
%cd /content/ppe
!python model/train/segformer_foodseg103.py \
  --model "{MODEL}" --epochs {EPOCHS} --batch-size {BATCH} \
  --output "{OUT}"

In [ ]:
# Result row for the README 'Testing set & results' table. Robust: reads the
# metric wherever it landed, never IndexErrors on missing checkpoint dirs.
import glob, json, os

def find_miou(out):
    p = os.path.join(out, 'eval_results.json')  # written by the training script
    if os.path.exists(p):
        m = json.load(open(p)).get('eval_mean_iou')
        if m is not None:
            return m
    states = [os.path.join(out, 'trainer_state.json')] + \
        sorted(glob.glob(f'{out}/checkpoint-*/trainer_state.json'))
    for sp in states:
        if not os.path.exists(sp):
            continue
        d = json.load(open(sp))
        if d.get('best_metric') is not None:
            return d['best_metric']
        ious = [e['eval_mean_iou'] for e in d.get('log_history', []) if 'eval_mean_iou' in e]
        if ious:
            return max(ious)
    return None

best = find_miou(OUT)
if best is None:
    print('No metric file found. Contents of OUT:')
    print(sorted(os.listdir(OUT)) if os.path.isdir(OUT) else 'OUT missing')
    print('If OUT holds model.safetensors the model trained fine — '
          'scroll the training cell for the last eval_mean_iou.')
else:
    print(f'best mIoU: {best:.4f}')
    print(f'| FoodSeg103 val | mIoU | public <= 0.05; B0 ~ 0.25, B1 ~ 0.32 | '
          f'**{best:.3f}** ({MODEL}) |')